# Lesson 04 — Counting and summarizing

In this lesson:
- Meet a new, bigger table: **`movies`** (35 rows)
- Count rows with `COUNT`
- Sum, average, min, max with `SUM`, `AVG`, `MIN`, `MAX`
- Round numbers with `ROUND`
- Give columns nicer names with `AS` (aliases)

Time: 25–30 minutes.

So far you've pulled rows out of a table. Today you start *summarizing* them — turning thousands of rows into a single answer. *"How many movies do we have?"* *"What's the average score?"* *"What was the highest-grossing one?"* That's the work of this lesson.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## Meet the `movies` table

The `disney_lessons` dataset now has a second table — `movies`. It has 35 rows, one per Disney/Pixar/Marvel/Lucasfilm movie, with these columns:

| Column | Type | Meaning |
|--------|------|---------|
| `title` | text | Movie title |
| `studio` | text | "Walt Disney Animation Studios", "Pixar", "Marvel Studios", or "Lucasfilm" |
| `year` | number | Release year |
| `genre` | text | "Animation" or "Action" |
| `runtime_minutes` | number | How long the movie runs, in minutes |
| `box_office_millions` | number | Worldwide box office, in millions of dollars |
| `rt_score` | number | Rotten Tomatoes score (0–100) |

Take a peek:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.movies
LIMIT 5

## What's an aggregate?

An **aggregate function** is a function that takes many rows of input and returns one number out.

The five most common ones, all built into SQL:

| Function | What it does |
|----------|--------------|
| `COUNT(...)` | How many rows |
| `SUM(...)` | Add up the values in a column |
| `AVG(...)` | Average of a column |
| `MIN(...)` | Smallest value in a column |
| `MAX(...)` | Largest value in a column |

You put them in the `SELECT` line, just like any other column. Let's start with `COUNT`.

## `COUNT` — how many?

`COUNT(*)` counts the rows in the result. The `*` is the same "everything" wildcard you've seen before — it tells `COUNT` "count every row, regardless of which columns are filled in."

How many movies are in our table?

In [ ]:
%%bigquery
SELECT COUNT(*)
FROM disney_lessons.movies

**1 row** comes back, with one cell: `35`. The result is itself a tiny table — even when it's just one number, SQL hands it back as a table.

Notice the column header is something ugly like `f0_`. That's the default name when you don't give the column a name. Let's fix that.

## `AS` — naming your columns

You can name (or rename) any column in the `SELECT` list using `AS`. It's called an **alias**:

In [ ]:
%%bigquery
SELECT COUNT(*) AS total_movies
FROM disney_lessons.movies

Same answer (`35`), but the column header is now `total_movies` — much friendlier.

You can use `AS` on any column, not just aggregates:

```sql
SELECT title AS film, year AS released
FROM disney_lessons.movies
```

## `SUM` — add up a column

`SUM` adds together all the values in a numeric column. What's the total box office across all our movies?

In [ ]:
%%bigquery
SELECT SUM(box_office_millions) AS total_box_office_millions
FROM disney_lessons.movies

You'll see a big number — somewhere around $25,000+ (millions). That's the lifetime box office of every movie in our table, summed together.

## `AVG` — average

`AVG` returns the arithmetic mean of a column. What's the average runtime?

In [ ]:
%%bigquery
SELECT AVG(runtime_minutes) AS avg_runtime
FROM disney_lessons.movies

Probably something like 105.something. Average runtime in minutes across all movies.

## `ROUND` — pretty up the decimals

The `AVG` result has a long decimal tail. To round it, wrap the whole expression in `ROUND(..., n)` where `n` is how many decimal places you want:

In [ ]:
%%bigquery
SELECT ROUND(AVG(runtime_minutes), 1) AS avg_runtime
FROM disney_lessons.movies

Same number, rounded to 1 decimal place. `ROUND(..., 0)` would round to a whole number.

## `MIN` and `MAX`

`MIN` is the smallest value, `MAX` is the largest. They work on numbers, dates, and even text (alphabetical order).

The oldest and newest movie in our table:

In [ ]:
%%bigquery
SELECT
  MIN(year) AS oldest_year,
  MAX(year) AS newest_year
FROM disney_lessons.movies

**1 row, 2 columns:** the oldest year (1937, Snow White) and the newest year (2023).

Note the formatting — multiple selected columns separated by commas, and we put each on its own line for readability. SQL doesn't care about whitespace; this is just for human eyes.

## Combining aggregates

You can put as many aggregates as you want in one `SELECT`:

In [ ]:
%%bigquery
SELECT
  COUNT(*) AS total_movies,
  ROUND(AVG(rt_score), 1) AS avg_score,
  MIN(rt_score) AS lowest_score,
  MAX(rt_score) AS highest_score
FROM disney_lessons.movies

**1 row, 4 columns** — a complete summary of how movies in our table score on Rotten Tomatoes.

## `WHERE` works with aggregates too

You can filter rows with `WHERE` *before* aggregating. The aggregate then runs only on the matching rows.

Average box office of just the Pixar movies:

In [ ]:
%%bigquery
SELECT
  COUNT(*) AS pixar_movies,
  ROUND(AVG(box_office_millions), 0) AS avg_box_office
FROM disney_lessons.movies
WHERE studio = 'Pixar'

## `COUNT(column)` vs. `COUNT(*)`

A subtle distinction worth knowing:

- `COUNT(*)` counts every row.
- `COUNT(some_column)` counts every row where `some_column` is **not NULL**.

For our movies table no columns are NULL, so they'd give the same answer. But this matters in tables where some columns can be missing — `COUNT(some_column)` becomes "how many rows have a value in this column."

We'll see this distinction matter in Lesson 08 when we work with the attractions table (which has `NULL` height limits).

## Quick reminders

- Read-only as always.
- `%%bigquery` is just Colab plumbing — drop the line in any other SQL tool.

## Exercises

### Exercise 1

How many **Marvel Studios** movies are in our table? Use `COUNT(*)` and a `WHERE` clause. Give the result column a friendly name with `AS`.

In [ ]:
# Your query here

### Exercise 2

What's the **total box office** of all our movies, in millions? Round to a whole number.

In [ ]:
# Your query here

### Exercise 3

Find the **highest** and **lowest** Rotten Tomatoes scores in our table, in one query. Give each its own clear column name.

In [ ]:
# Your query here

### Exercise 4

What's the **average runtime** of action movies (`genre = 'Action'`), rounded to 1 decimal place?

In [ ]:
# Your query here

---

## Solutions

⚠️ **Spoilers below.**

### Solution 1

In [ ]:
%%bigquery
SELECT COUNT(*) AS marvel_movies
FROM disney_lessons.movies
WHERE studio = 'Marvel Studios'

### Solution 2

In [ ]:
%%bigquery
SELECT ROUND(SUM(box_office_millions), 0) AS total_box_office_millions
FROM disney_lessons.movies

### Solution 3

In [ ]:
%%bigquery
SELECT
  MAX(rt_score) AS highest_score,
  MIN(rt_score) AS lowest_score
FROM disney_lessons.movies

### Solution 4

In [ ]:
%%bigquery
SELECT ROUND(AVG(runtime_minutes), 1) AS avg_action_runtime
FROM disney_lessons.movies
WHERE genre = 'Action'

## What you've learned

- ✅ `COUNT(*)` to count rows
- ✅ `SUM`, `AVG`, `MIN`, `MAX` to summarize columns
- ✅ `ROUND` for clean decimals
- ✅ `AS` to give columns friendly names
- ✅ Combining `WHERE` with aggregates

## Up next

Right now, every aggregate query collapses the entire table into a single row. But what if you want one number per studio? Or per year? Or per genre?

That's `GROUP BY` — the second-most-powerful feature in SQL (after joins). Open `05_grouping.ipynb`.